In [ ]:
!pip install ultralytics
from ultralytics import YOLO
import cv2
import numpy as np
import time

modelo = YOLO("yolov8m.pt")

In [ ]:
# area da fila
ROI_X1 = 100
ROI_Y1 = 100
ROI_X2 = 900
ROI_Y2 = 700

In [ ]:
def contar_pessoas(imagem):
  inicio = time.time()

  resultados = modelo.track(
      source = imagem,
      classes=[0],
      persist=True,
      conf=0.55,
      imgsz = 1280
  )

  tempo_execucao = time.time() - inicio

  frame_saida = resultados[0].plot()

  cv2.rectangle(
      frame_saida,
      (ROI_X1, ROI_Y1),
      (ROI_X2, ROI_Y2),
      (255, 0, 0),
      2
  )

  ids_contados = set()
  confiancas = []

  if resultados[0].boxes.id is not None:
    ids = resultados[0].boxes.id.cpu().numpy()
    boxes = resultados[0].boxes.xyxy.cpu().numpy()
    confs = resultados[0].boxes.conf.cpu().numpy()

    for box, track_id, conf in zip(boxes, ids, confs):
      x1, y1, x2, y2 = box
      cx, cy = (x1 + x2) / 2, (y1 + y2) / 2

      if ROI_X1 <= cx <= ROI_X2 and ROI_Y1 <= cy <= ROI_Y2:
        ids_contados.add(int(track_id))
        confiancas.append(float(conf))

  quantidade = len(ids_contados)

  estatisticas = {
      "tempo_execucao": tempo_execucao,
      "quantidade": quantidade,
      "confiancas": confiancas
  }

  return quantidade, frame_saida, estatisticas

In [ ]:
from ultralytics import YOLO

modelo = YOLO("yolov8m.pt")

modelo.export(format="onnx")